# Regime Classifier — Live MEXC Validation

Fetches real 1h candles from MEXC (no API key needed for public data), runs the ADX+EMA+BB classifier, and plots the regime labels overlaid on price so we can sanity-check them visually.

In [ ]:
import sys
sys.path.insert(0, '..')   # so imports resolve from project root

import ccxt
import pandas as pd
import pandas_ta as ta
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from loguru import logger
logger.disable('')  # suppress loguru noise in notebook output

from regime.classifier import classify_single, _to_df
from utils.config import get as cfg

%matplotlib inline
plt.rcParams['figure.figsize'] = (16, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print('imports ok')

In [ ]:
# ── Config ───────────────────────────────────────────────────────────────────
PAIRS     = ['BTC/USDT:USDT', 'ETH/USDT:USDT', 'SOL/USDT:USDT']
TIMEFRAME = '1h'
LIMIT     = 300   # candles to fetch (≈12 days of 1h data)

REGIME_COLORS = {
    'trending-up':   '#2ecc71',   # green
    'trending-down': '#e74c3c',   # red
    'choppy':        '#95a5a6',   # grey
    'high-volatility': '#f39c12', # orange
    'unknown':       '#ecf0f1',
}

In [ ]:
# ── Fetch candles from MEXC (public endpoint — no API key required) ───────────
exchange = ccxt.mexc({'options': {'defaultType': 'swap'}})

raw: dict[str, list] = {}
for pair in PAIRS:
    ohlcv = exchange.fetch_ohlcv(pair, TIMEFRAME, limit=LIMIT)
    raw[pair] = ohlcv
    print(f'{pair}: {len(ohlcv)} candles  '
          f'from {pd.to_datetime(ohlcv[0][0], unit="ms").strftime("%Y-%m-%d %H:%M")}'
          f' to {pd.to_datetime(ohlcv[-1][0], unit="ms").strftime("%Y-%m-%d %H:%M")}')

In [ ]:
# ── Label every candle with a regime (rolling window) ────────────────────────
c   = cfg()['regime']
MIN = c['ema_slow'] + 5  # minimum rows needed by classify_single

def label_all(ohlcv: list) -> pd.DataFrame:
    df = _to_df(ohlcv)
    regimes, confs = [], []
    for i in range(len(df)):
        if i < MIN:
            regimes.append('unknown')
            confs.append(float('nan'))
        else:
            try:
                lbl, conf = classify_single(df.iloc[:i + 1])
                regimes.append(lbl)
                confs.append(conf)
            except Exception:
                regimes.append('unknown')
                confs.append(float('nan'))
    df['regime']     = regimes
    df['confidence'] = confs
    return df

labeled: dict[str, pd.DataFrame] = {}
for pair, ohlcv in raw.items():
    labeled[pair] = label_all(ohlcv)
    dist = labeled[pair]['regime'].value_counts().to_dict()
    print(f'{pair}: {dist}')

In [ ]:
# ── Plot: price + EMA20/50 + regime background shading ───────────────────────
def plot_pair(pair: str, df: pd.DataFrame) -> None:
    cr = cfg()['regime']
    ema_fast = ta.ema(df['close'], length=cr['ema_fast'])
    ema_slow = ta.ema(df['close'], length=cr['ema_slow'])

    fig, axes = plt.subplots(3, 1, figsize=(18, 10),
                             gridspec_kw={'height_ratios': [3, 1, 1]},
                             sharex=True)
    fig.suptitle(pair, fontsize=14, fontweight='bold')

    ax_price, ax_adx, ax_bb = axes

    # ── Price + EMA + regime background ──────────────────────────────────────
    ax_price.plot(df.index, df['close'], color='#2c3e50', lw=1.2, label='Close')
    ax_price.plot(df.index, ema_fast, color='#3498db', lw=1,
                  linestyle='--', label=f'EMA{cr["ema_fast"]}')
    ax_price.plot(df.index, ema_slow, color='#e67e22', lw=1,
                  linestyle='--', label=f'EMA{cr["ema_slow"]}')

    # shade background by regime
    prev_regime, start = df['regime'].iloc[0], df.index[0]
    for ts, row in df.iterrows():
        if row['regime'] != prev_regime:
            ax_price.axvspan(start, ts, alpha=0.15,
                             color=REGIME_COLORS.get(prev_regime, '#ffffff'))
            start, prev_regime = ts, row['regime']
    ax_price.axvspan(start, df.index[-1], alpha=0.15,
                     color=REGIME_COLORS.get(prev_regime, '#ffffff'))

    ax_price.set_ylabel('Price (USDT)')
    legend_patches = [
        mpatches.Patch(color=v, alpha=0.6, label=k)
        for k, v in REGIME_COLORS.items() if k != 'unknown'
    ]
    ax_price.legend(handles=[
        plt.Line2D([0], [0], color='#2c3e50', lw=1.2, label='Close'),
        plt.Line2D([0], [0], color='#3498db', lw=1, ls='--', label=f'EMA{cr["ema_fast"]}'),
        plt.Line2D([0], [0], color='#e67e22', lw=1, ls='--', label=f'EMA{cr["ema_slow"]}'),
    ] + legend_patches, loc='upper left', fontsize=8)

    # ── ADX subplot ──────────────────────────────────────────────────────────
    adx_df = ta.adx(df['high'], df['low'], df['close'], length=cr['adx_period'])
    adx_col = f'ADX_{cr["adx_period"]}'
    ax_adx.plot(df.index, adx_df[adx_col], color='#8e44ad', lw=1, label='ADX')
    ax_adx.axhline(cr['adx_trend_threshold'], color='#8e44ad',
                   lw=0.8, linestyle=':', label=f'threshold={cr["adx_trend_threshold"]}')
    ax_adx.set_ylabel('ADX')
    ax_adx.legend(fontsize=8, loc='upper left')
    ax_adx.set_ylim(0, None)

    # ── BB Width subplot ──────────────────────────────────────────────────────
    bb = ta.bbands(df['close'], length=cr['bb_period'], std=cr['bb_std'])
    bb_col = next(col for col in bb.columns if col.startswith('BBB_'))
    bb_width = bb[bb_col]
    bb_avg   = bb_width.rolling(20).mean()
    ax_bb.plot(df.index, bb_width, color='#f39c12', lw=1, label='BB Width')
    ax_bb.plot(df.index, bb_avg * cr['high_vol_bb_multiplier'],
               color='#e74c3c', lw=0.8, linestyle=':',
               label=f'avg × {cr["high_vol_bb_multiplier"]} (HV trigger)')
    ax_bb.set_ylabel('BB Width')
    ax_bb.legend(fontsize=8, loc='upper left')

    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

for pair, df in labeled.items():
    plot_pair(pair, df)

In [ ]:
# ── Regime distribution table ─────────────────────────────────────────────────
rows = []
for pair, df in labeled.items():
    knowns = df[df['regime'] != 'unknown']
    total  = len(knowns)
    for regime in ['trending-up', 'trending-down', 'choppy', 'high-volatility']:
        n   = (knowns['regime'] == regime).sum()
        pct = n / total * 100 if total else 0
        avg_conf = knowns.loc[knowns['regime'] == regime, 'confidence'].mean()
        rows.append({'pair': pair, 'regime': regime, 'candles': n,
                     'pct': f'{pct:.1f}%', 'avg_confidence': f'{avg_conf:.3f}' if not np.isnan(avg_conf) else 'N/A'})

summary = pd.DataFrame(rows)
summary

In [ ]:
# ── Spot-check: last 5 candle labels for each pair ───────────────────────────
for pair, df in labeled.items():
    print(f'\n{pair} — last 5 candles:')
    print(df[['close', 'regime', 'confidence']].tail(5).to_string())

In [ ]:
# ── Current regime for each pair (what the engine would classify right now) ───
print('Current regime snapshot:')
print('-' * 55)
for pair, df in labeled.items():
    last = df[df['regime'] != 'unknown'].iloc[-1]
    print(f"  {pair:<22} {last['regime']:<18} conf={last['confidence']:.3f}")